# NutriEpiDB: A Database For Dietary Compounds With Epigenetic Targets Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the NutriEpiDB dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema URL and contains tabular data from literature on dietary compounds and their epigenetic targets.

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load the NutriEpiDB metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.ezq8-60vy/fair2.json'

# Load the dataset from the Croissant schema
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published {metadata.datePublished}, Version {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values. Use `mlcroissant` to explore the dataset entities via their `@id`s.

In [ ]:
# Retrieve RecordSet objects and print overview using their @ids
record_sets = dataset.record_sets
print(f"Available record sets ({len(record_sets)}):")
for rs in record_sets:
    print(f"  @id: {rs.id} | name: {rs.name} | description: {rs.description}")
    # Print fields for this record set
    print("    Fields:")
    for field in rs.fields:
        print(f"      @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    print()

# For demonstration, show 3 records from the first record set (if available)
if record_sets:
    first_rs_id = record_sets[0].id
    print(f"First three records from record set '@id': {first_rs_id}")
    for i, record in enumerate(dataset.records(record_set=first_rs_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load tabular data from each record set into Pandas DataFrames for further analysis.
All record sets, fields, and columns are referenced strictly by their `@id` values.

In [ ]:
# Extract all main record sets into DataFrames, using @id as keys
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    if not df.empty:
        print(f"Record set '@id': {rs_id} columns:")
        print(df.columns.tolist())
        print(df.head(2), "\n")
    dataframes[rs_id] = df

# Choose main record set (e.g. the first by default)
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id:
    print(f"Columns in main record set '@id': {main_rs_id}")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process the main record set: filter records, normalize numeric fields, and group by categorical fields.

Identify fields by their `@id`, and perform basic EDA and transformations.

In [ ]:
# For demonstration, use numeric and categorical fields from first record set
main_rs = dataset.record_sets[0] if dataset.record_sets else None
df = dataframes[main_rs_id] if main_rs_id else pd.DataFrame()

# Find a numeric field @id for EDA
numeric_field_id = None
group_field_id = None
for field in main_rs.fields:
    if field.data_type in ['Integer', 'Float', 'Number'] and not numeric_field_id:
        numeric_field_id = field.id
    if field.data_type in ['Text', 'String'] and not group_field_id:
        group_field_id = field.id

print(f"Numeric field for EDA: {numeric_field_id}")
print(f"Grouping field for EDA: {group_field_id}")

# If numeric field exists, filter, normalize, and group
if numeric_field_id in df.columns:
    threshold = df[numeric_field_id].quantile(0.90) if df[numeric_field_id].dtype != 'O' else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where field @id '{numeric_field_id}' > {threshold}:")
    print(filtered_df.head())

    # Add normalized column
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].dropna()].head())
else:
    print("No numeric field detected for EDA.")

# Group by a categorical field if exists
if group_field_id and group_field_id in df.columns and numeric_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped filtered data by field @id '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships of numeric and categorical fields in the dataset, always referencing fields by their `@id` values.

In [ ]:
# Basic visualization using matplotlib and pandas
import matplotlib.pyplot as plt

# Plot histogram for numeric field if available
if numeric_field_id and numeric_field_id in df.columns:
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of field '@id': {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Plot bar chart of grouped means
if group_field_id and group_field_id in df.columns and numeric_field_id in df.columns:
    grouped_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
    grouped_means.head(10).plot.bar()
    plt.title(f"Top 10 '{group_field_id}' by mean of field '@id': {numeric_field_id}")
    plt.ylabel('Mean value')
    plt.xlabel(group_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring NutriEpiDB using `mlcroissant`, referencing all dataset entities by their `@id` values for reproducibility. The basic EDA and visualizations give insight into the data's structure and potential analyses.

Continue your exploration by applying domain-specific filters or deep learning models on these curated biomedical relationships.